# LM2500 — Tier B: GGOV1 governor replacement

Replaces the TGOV1 reheat-steam-turbine model (a poor fit for an aero free-PT gas turbine) with **GGOV1** from IEEE PES-TR1 (2013), Fig 3-5.

GGOV1 captures the three principal LM2500 control loops:

1. **Speed/power governor** — PI on speed error with electrical-power droop feedback
2. **Acceleration controller** — integral on filtered angular acceleration, prevents compressor surge
3. **Temperature/load limiter** — PI on a fuel-flow-derived temperature proxy, clips output

arbitrated through a **low-value-select gate** with back-calculation anti-windup on the two unselected integrators.

Validation strategy:

- **Cell 4** — Beluga 5 load rejection (Hannett & Khan 1993 Figs 4, 5): overlay turbine speed and V_ce against digitized field measurements, using PES-TR1 defaults (heavy-duty CT, matches Beluga 5).
- **Cell 5** — Hannett Table 3 metric: 50→60 % step, time to reach 0.6 Pm. Target: 2.32 s derived (vs 1.14 s typical-model).
- **Cell 6** — Re-run Load17 7.4-hr profile with LM2500-tuned GGOV1. Compare to Tier A TGOV1.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gas_plant import GasTurbinePlant
from gas_plant.dynamics import GGOV1Params, simulate_ggov1, step_response
from gas_plant.dynamics import TGOV1Params, simulate_tgov1  # for Tier A comparison

lm2500 = GasTurbinePlant(rated_power_mw=22.0)
dispatch_fn = lambda lf: lm2500.dispatch(lf)
print('Imports OK')

## 1. Parameter sets

Two GGOV1 parameter sets:

- **`p_pes`** — pure PES-TR1 Appendix C defaults (GE heavy-duty CT, ~Beluga 5)
- **`p_lm2500`** — LM2500 overrides (faster Woodward MkVIe actuator)

Differences (`p_lm2500` vs `p_pes`):

| Param | PES-TR1 default | LM2500 override | Rationale |
|---|---|---|---|
| TACT | 0.5 s | 0.15 s | LM2500 Woodward MkVIe electronic valve actuator |
| Vmax | 1.0 pu | 22/23 ≈ 0.957 pu | rebased to turbine MW on 23 MVA gen base (Tier A A4) |

In [ ]:
p_pes = GGOV1Params()                       # straight PES-TR1 Appendix C
p_lm2500 = GGOV1Params.lm2500_overrides()   # LM2500-tuned overrides

from dataclasses import fields
deltas = []
for f in fields(GGOV1Params):
    a, b = getattr(p_pes, f.name), getattr(p_lm2500, f.name)
    if a != b:
        deltas.append({'param': f.name, 'PES-TR1': a, 'LM2500': b})
pd.DataFrame(deltas)

## 2. Steady-state check

Hold demand at 15 MW for 30 s. Confirm no drift in either parameter set.

In [ ]:
for label, p in [('PES-TR1 defaults', p_pes), ('LM2500 overrides', p_lm2500)]:
    r = simulate_ggov1(np.array([0.0, 30.0]), np.array([15.0, 15.0]),
                       params=p, sample_dt_s=1.0)
    print(f'{label:24s} df={r.freq_hz[-1]-60:+.6f} Hz   '
          f'Pm={r.Pm_mw[-1]:.4f} MW   valve={r.valve_pu[-1]:.4f}')

## 3. Beluga 5 load rejection — overlay against Hannett Figs 4 & 5

**Test:** Unit at 6 MW load, sudden trip to 0 MW load. Track turbine speed (Fig 4) and V_ce (Fig 5).

Using **`p_pes`** (PES-TR1 defaults) because Beluga 5 is a heavy-duty Alaskan CT, not an aero. 

Hannett x-axis is in 60 Hz cycles → divide by 60 for seconds.

In [ ]:
fig4 = pd.read_csv(ROOT / 'papers/hannett_fig4_digitized.csv', header=None,
                   names=['cycles', 'd_omega_pu'])
fig5 = pd.read_csv(ROOT / 'papers/hannett_figure5.csv', header=None,
                   names=['cycles', 'V_ce_pu'])
fig4['t_s'] = fig4['cycles'] / 60.0
fig5['t_s'] = fig5['cycles'] / 60.0
print(f'Fig 4: {len(fig4)} points, t∈[{fig4["t_s"].min():.2f}, {fig4["t_s"].max():.2f}] s, '
      f'Δω∈[{fig4["d_omega_pu"].min():.4f}, {fig4["d_omega_pu"].max():.4f}]')
print(f'Fig 5: {len(fig5)} points, t∈[{fig5["t_s"].min():.2f}, {fig5["t_s"].max():.2f}] s, '
      f'V_ce∈[{fig5["V_ce_pu"].min():.4f}, {fig5["V_ce_pu"].max():.4f}]')

# Build a Beluga-5-like machine. Beluga 5 ~ 6 MW unit on small gen.
# We'll use Sn=6 MVA (close to 1:1 turbine:gen rating) with PES-TR1 dynamics.
# Initial load 6 MW (full load), then trip to 0 MW at t=2 s.
p_beluga = GGOV1Params(
    Sn_mva=6.0, P_turbine_mw=6.0, H_s=2.0, D_pu=1.0,
    # PES-TR1 defaults for everything else (heavy-duty)
    Vmax_pu=1.0,  # = turbine MW / gen MVA = 6/6
    alpha_load_damping=0.0,   # constant-power load rejection (load goes to zero)
)

# Load rejection: 6 MW down to 0.01 MW (tiny non-zero so dispatch math behaves)
load_t = np.array([0.0, 2.0, 16.0])
load_mw = np.array([6.0, 0.01, 0.01])
res_rej = simulate_ggov1(load_t, load_mw, params=p_beluga, sample_dt_s=0.01)

# Align our t=0 with Hannett t=0 (which appears to be just before the trip).
# Hannett Fig 4 shows speed deviation ramping up starting ~at cycle 117 (1.95 s).
# Shift our time so the trip-time-offset matches.
t_model = res_rej.t_s - 2.0 + 117/60  # align trip event at Hannett's t=117/60≈1.95s

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
ax = axes[0]
ax.plot(fig4['t_s'], fig4['d_omega_pu'], 'ko-', markersize=4, label='Hannett Fig 4 (Beluga 5 measured)')
ax.plot(t_model, res_rej.omega_pu - 1.0, 'C0-', lw=1.5, label='GGOV1 (PES-TR1 defaults)')
ax.set_ylabel('Δω [pu]'); ax.set_title('Beluga 5 6 MW load rejection — turbine speed deviation')
ax.legend(); ax.grid(alpha=0.3)
ax.axhline(0, color='gray', ls=':', lw=0.7)

ax = axes[1]
ax.plot(fig5['t_s'], fig5['V_ce_pu'], 'ko-', markersize=4, label='Hannett Fig 5 (Beluga 5 measured)')
ax.plot(t_model, res_rej.fsr_pu, 'C3-', lw=1.5, label='GGOV1 fsr (= V_ce equivalent)')
ax.set_ylabel('V_ce [pu]'); ax.set_xlabel('Time [s]'); ax.legend(); ax.grid(alpha=0.3)
ax.axhline(0, color='gray', ls=':', lw=0.7)
ax.set_title('Beluga 5 6 MW load rejection — V_ce (fuel demand)')

fig.tight_layout()
plt.show()

# Quantitative comparison
peak_meas = fig4['d_omega_pu'].max()
settle_meas = fig4['d_omega_pu'].iloc[-1]
post_trip = res_rej.t_s > 2.0
peak_model = (res_rej.omega_pu[post_trip] - 1.0).max()
settle_model = res_rej.omega_pu[-1] - 1.0
print('Speed-deviation peak / settle comparison:')
print(f'  measured: peak={peak_meas:+.4f} pu, settle={settle_meas:+.4f} pu')
print(f'  model:    peak={peak_model:+.4f} pu, settle={settle_model:+.4f} pu')

## 4. Hannett Table 3 metric — time-to-0.6-Pm after 50 → 60 % step

**Test setup (per Hannett 1993 §Comparison):**
- Initial load = 50 % of generator MVA
- Disturbance: step increase of 10 % of generator MVA
- Metric: time for Pm to reach 0.6 pu (= 60 % of generator MVA)

**Beluga 5 target:** 2.32 s (derived from field test) vs 1.14 s (typical-model)

Our LM2500 model should fall in the 2.0–2.6 s range when using PES-TR1 defaults (since Beluga 5 is a heavy-duty CT). With LM2500-aero overrides it should be faster.

In [ ]:
def time_to_60pct(params, Sn=23.0, t_step=2.0, t_end=15.0):
    P_init = 0.5 * Sn
    P_final = 0.6 * Sn
    r = step_response(P_init, P_final, t_step_s=t_step, t_end_s=t_end,
                      sample_dt_s=0.005, params=params)
    threshold_mw = 0.6 * Sn  # Pm at 0.6 pu of MVA
    above = np.where((r.t_s >= t_step) & (r.Pm_mw >= threshold_mw))[0]
    if len(above) == 0:
        # never reached due to droop offset — try 0.99 of new steady state
        target_pm = r.Pm_mw[-1] * 0.99
        above = np.where((r.t_s >= t_step) & (r.Pm_mw >= target_pm))[0]
        t_threshold = r.t_s[above[0]] - t_step if len(above) > 0 else np.nan
        return t_threshold, r, 'fallback: 99% of SS'
    return r.t_s[above[0]] - t_step, r, 'direct: 0.6 of MVA'

# Run both parameter sets
for label, p in [('PES-TR1 defaults (~Beluga 5)', p_pes),
                  ('LM2500 overrides (aero)',       p_lm2500)]:
    t60, r, mode = time_to_60pct(p, Sn=p.Sn_mva)
    Pm_ss = r.Pm_mw[-1]
    print(f'{label}: time to reach 0.6 Pm = {t60:.3f} s   '
          f'(Pm_SS={Pm_ss:.3f} MW, freq_SS={r.freq_hz[-1]:.4f} Hz, mode={mode})')
print()
print('Hannett 1993 Table 3, Beluga 5:')
print('  Typical model with 3% droop: 1.140 s')
print('  Derived from field test:     2.320 s')

In [ ]:
# Plot the step response for each parameter set
t_step = 2.0
r_pes = step_response(0.5*23.0, 0.6*23.0, t_step_s=t_step, t_end_s=15.0, sample_dt_s=0.005, params=p_pes)
r_lm = step_response(0.5*23.0, 0.6*23.0, t_step_s=t_step, t_end_s=15.0, sample_dt_s=0.005, params=p_lm2500)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
ax = axes[0]
ax.plot(r_pes.t_s - t_step, r_pes.Pm_mw, 'C0-', lw=1.5, label='PES-TR1 defaults')
ax.plot(r_lm.t_s - t_step, r_lm.Pm_mw, 'C2-', lw=1.5, label='LM2500 overrides')
ax.axhline(0.6*23.0, color='k', ls='--', lw=1, label='0.6 Pm target = 13.8 MW')
ax.axhline(0.5*23.0, color='gray', ls=':', lw=1, label='initial Pm = 11.5 MW')
ax.set_ylabel('Pm [MW]'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('50 → 60 % step response (Hannett Table 3 protocol)')

ax = axes[1]
ax.plot(r_pes.t_s - t_step, r_pes.freq_hz, 'C0-', lw=1.5, label='PES-TR1')
ax.plot(r_lm.t_s - t_step, r_lm.freq_hz, 'C2-', lw=1.5, label='LM2500')
ax.axhline(60.0, color='k', ls=':', lw=0.7)
ax.set_ylabel('Frequency [Hz]'); ax.set_xlabel('Time after step [s]'); ax.legend(); ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 5. Load17 7.4-hr re-run — GGOV1 vs Tier A TGOV1

Same demand profile, three models side by side:

- Tier 0 baseline (original TGOV1, all fixes off)
- Tier A (TGOV1 + 6 bug fixes)
- Tier B (GGOV1 with LM2500 overrides)

In [ ]:
load17 = pd.read_csv(ROOT / 'data' / 'load17.csv', sep=r'\s+', header=None, names=['time_s','demand_raw'])
load17['demand_mw'] = load17['demand_raw'] * 10.0
t_arr = load17['time_s'].values
d_arr = load17['demand_mw'].values

# Tier 0 baseline
params_t0 = TGOV1Params(use_anti_windup=False, use_valve_rate_limit=False,
                        alpha_load_damping=0.0, fuel_from_valve=False, chunked=False,
                        vmax_pu=1.1, vmin_pu=0.0, T_comb_s=0.01)
res_t0 = simulate_tgov1(t_arr, d_arr, params=params_t0, sample_dt_s=1.0, dispatch_fn=dispatch_fn)

# Tier A
res_tA = simulate_tgov1(t_arr, d_arr, params=TGOV1Params(), sample_dt_s=1.0, dispatch_fn=dispatch_fn)

# Tier B (LM2500 overrides)
res_tB = simulate_ggov1(t_arr, d_arr, params=p_lm2500, sample_dt_s=1.0, dispatch_fn=dispatch_fn)

for label, r in [('Tier 0', res_t0), ('Tier A', res_tA), ('Tier B (GGOV1)', res_tB)]:
    dfmax = max(abs(r.freq_hz.max()-60), abs(60-r.freq_hz.min()))*1000
    fuel_t = (r.cum_fuel_kg[-1]/1000) if r.cum_fuel_kg is not None else float('nan')
    print(f'{label:18s} |Δf|max = {dfmax:6.1f} mHz   fuel = {fuel_t:.3f} t')

In [ ]:
t_hr = res_tB.t_s / 3600
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

ax = axes[0]
ax.plot(t_hr, res_tB.Pe_mw, 'C3-', lw=0.4, alpha=0.7, label='Demand (Load17)')
ax.plot(t_hr, res_t0.Pm_mw, 'k:', lw=0.6, label='Pm Tier-0')
ax.plot(t_hr, res_tA.Pm_mw, 'C2-', lw=0.5, label='Pm Tier-A (TGOV1+fixes)')
ax.plot(t_hr, res_tB.Pm_mw, 'C0-', lw=0.5, label='Pm Tier-B (GGOV1)')
ax.set_ylabel('Power [MW]'); ax.legend(loc='upper right', fontsize=9); ax.grid(alpha=0.3)
ax.set_title('LM2500 islanded transient — three-model comparison (Load17, 7.4 hr)')

ax = axes[1]
ax.plot(t_hr, res_t0.freq_hz, 'k:', lw=0.4, label='Tier-0')
ax.plot(t_hr, res_tA.freq_hz, 'C2-', lw=0.4, label='Tier-A')
ax.plot(t_hr, res_tB.freq_hz, 'C0-', lw=0.4, label='Tier-B')
ax.axhline(60.0, color='gray', ls=':', lw=1)
ax.set_ylabel('Frequency [Hz]'); ax.legend(loc='upper right', fontsize=9); ax.grid(alpha=0.3)

ax = axes[2]
ax.plot(t_hr, res_tA.P_valve_pu, 'C2-', lw=0.4, label='Tier-A valve')
ax.plot(t_hr, res_tB.valve_pu, 'C0-', lw=0.4, label='Tier-B valve')
ax.set_ylabel('Valve [pu]'); ax.legend(loc='upper right', fontsize=9); ax.grid(alpha=0.3)

ax = axes[3]
ax.plot(t_hr, res_t0.cum_fuel_kg/1000, 'k:', lw=1.0, label='Tier-0')
ax.plot(t_hr, res_tA.cum_fuel_kg/1000, 'C2-', lw=1.0, label='Tier-A')
ax.plot(t_hr, res_tB.cum_fuel_kg/1000, 'C0-', lw=1.0, label='Tier-B')
ax.set_ylabel('Cum fuel [t]'); ax.set_xlabel('Time [hours]'); ax.legend(loc='upper left', fontsize=9); ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 6. Which controller commanded the valve?

GGOV1's LVG arbitrates between speed governor (fsrn), acceleration controller (fsra), and temperature limiter (fsrt). For Load17 (always 65–91 % of rating, small ramps), we expect fsrn to dominate. Confirm.

In [ ]:
# Which controller is in command at each sample?
eps = 1e-6
by_speed = (res_tB.fsrn_pu <= res_tB.fsra_pu + eps) & (res_tB.fsrn_pu <= res_tB.fsrt_pu + eps)
by_accel = (res_tB.fsra_pu < res_tB.fsrn_pu - eps) & (res_tB.fsra_pu <= res_tB.fsrt_pu + eps)
by_temp  = (res_tB.fsrt_pu < res_tB.fsrn_pu - eps) & (res_tB.fsrt_pu < res_tB.fsra_pu - eps)
print('LVG share over Load17 (% of time each controller commands valve):')
print(f'  speed governor (fsrn):   {by_speed.mean()*100:.2f} %')
print(f'  acceleration ctrl (fsra):{by_accel.mean()*100:.2f} %')
print(f'  temperature ltd (fsrt):  {by_temp.mean()*100:.2f} %')

# Plot the three controller signals over a short window for inspection
mask = (res_tB.t_s >= 13800) & (res_tB.t_s <= 13900)
fig, ax = plt.subplots(1, 1, figsize=(12, 5))
ax.plot(res_tB.t_s[mask], res_tB.fsrn_pu[mask], 'C0-', lw=1.0, label='fsrn (speed gov)')
ax.plot(res_tB.t_s[mask], res_tB.fsra_pu[mask], 'C1-', lw=1.0, label='fsra (accel ctrl)')
ax.plot(res_tB.t_s[mask], res_tB.fsrt_pu[mask], 'C3-', lw=1.0, label='fsrt (temp limiter)')
ax.plot(res_tB.t_s[mask], res_tB.fsr_pu[mask], 'k--', lw=1.5, label='fsr (LVG selected)')
ax.plot(res_tB.t_s[mask], res_tB.valve_pu[mask], 'C2-', lw=1.5, alpha=0.7, label='valve')
ax.set_xlabel('Time [s]'); ax.set_ylabel('Signal [pu]')
ax.set_title(f'GGOV1 controller signals near largest Load17 ramp (event at t≈13815 s)')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 7. Summary

Captured in `docs/tier_plan.md` Tier B section.